In [1]:
from google.colab import drive
drive.mount ('/content/drive')

Mounted at /content/drive


In [7]:
import zipfile
import os

# Create dataset directories
os.makedirs('/content/dataset/yolo', exist_ok=True)
os.makedirs('/content/dataset/coco', exist_ok=True)
os.makedirs('/content/dataset/voc', exist_ok=True)

# Unzip each dataset
with zipfile.ZipFile('/content/drive/MyDrive/RD2/yolo_dataset.zip', 'r') as z:
    z.extractall('/content/dataset/yolo')

with zipfile.ZipFile('/content/drive/MyDrive/RD2/coco_dataset.zip', 'r') as z:
    z.extractall('/content/dataset/coco')

with zipfile.ZipFile('/content/drive/MyDrive/RD2/voc_dataset.zip', 'r') as z:
    z.extractall('/content/dataset/voc')

print("Datasets extracted successfully")

Datasets extracted successfully


In [4]:
!pip install ultralytics scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.4 MB/s eta 0:00:00


In [5]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")

GPU available: True
GPU name: NVIDIA A100-SXM4-80GB


In [8]:
import os

yolo_images = '/content/dataset/yolo/yolo/Smartphone-Detection-3/train/images'
print(f"YOLO images: {len(os.listdir(yolo_images))}")

YOLO images: 157


In [9]:
import os
import shutil
import yaml
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from sklearn.model_selection import KFold

# ─── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_DIR = "/content/dataset/yolo/yolo/Smartphone-Detection-3/train"
K_FOLDS     = 5
EPOCHS      = 50
IMG_SIZE    = 640
MODEL       = "yolov8m.pt"
RESULTS_DIR = "/content/results/yolo"
DRIVE_DIR   = "/content/drive/MyDrive/RD2/models/yolo"
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

images_dir = Path(DATASET_DIR) / "images"
labels_dir = Path(DATASET_DIR) / "labels"

image_files = sorted([
    f for f in images_dir.iterdir()
    if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

print(f"Total images found: {len(image_files)}")

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(image_files), 1):
    print(f"\n{'='*40}")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"  Train: {len(train_idx)} images")
    print(f"  Test:  {len(val_idx)} images")
    print(f"{'='*40}")

    # Create temp fold directories
    fold_dir = Path(f"/content/fold_{fold}")
    for split in ["train", "val"]:
        (fold_dir / split / "images").mkdir(parents=True, exist_ok=True)
        (fold_dir / split / "labels").mkdir(parents=True, exist_ok=True)

    # Copy files into fold directories
    for idx in train_idx:
        img = image_files[idx]
        lbl = labels_dir / (img.stem + ".txt")
        shutil.copy(img, fold_dir / "train" / "images" / img.name)
        if lbl.exists():
            shutil.copy(lbl, fold_dir / "train" / "labels" / lbl.name)

    for idx in val_idx:
        img = image_files[idx]
        lbl = labels_dir / (img.stem + ".txt")
        shutil.copy(img, fold_dir / "val" / "images" / img.name)
        if lbl.exists():
            shutil.copy(lbl, fold_dir / "val" / "labels" / lbl.name)

    # Write fold-specific yaml
    fold_yaml = {
        "train": str((fold_dir / "train" / "images").resolve()),
        "val":   str((fold_dir / "val"   / "images").resolve()),
        "nc":    2,
        "names": ["person", "smartphone"]
    }
    fold_yaml_path = f"/content/fold_{fold}.yaml"
    with open(fold_yaml_path, "w") as f:
        yaml.dump(fold_yaml, f)

    # Train
    model = YOLO(MODEL)
    model.train(
        data=fold_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        project=RESULTS_DIR,
        name=f"fold_{fold}",
        verbose=False,
        device=0
    )

    # Validate
    metrics = model.val(verbose=False)
    fold_map50     = float(metrics.box.map50)
    fold_map50_95  = float(metrics.box.map)
    fold_precision = float(metrics.box.mp)
    fold_recall    = float(metrics.box.mr)

    fold_results.append({
        "fold":      fold,
        "mAP50":     fold_map50,
        "mAP50-95":  fold_map50_95,
        "precision": fold_precision,
        "recall":    fold_recall
    })

    print(f"\nFold {fold} Results:")
    print(f"  mAP50:     {fold_map50:.4f}")
    print(f"  mAP50-95:  {fold_map50_95:.4f}")
    print(f"  Precision: {fold_precision:.4f}")
    print(f"  Recall:    {fold_recall:.4f}")

    # ─── Save model to Google Drive ───────────────────────────────────────────
    best_src = f"{RESULTS_DIR}/fold_{fold}/weights/best.pt"
    best_dst = f"{DRIVE_DIR}/yolo_fold{fold}_best.pt"
    last_src = f"{RESULTS_DIR}/fold_{fold}/weights/last.pt"
    last_dst = f"{DRIVE_DIR}/yolo_fold{fold}_last.pt"

    if os.path.exists(best_src):
        shutil.copy(best_src, best_dst)
        print(f"  ✅ Saved best.pt for fold {fold} to Drive")
    else:
        print(f"  ❌ best.pt not found for fold {fold}")

    if os.path.exists(last_src):
        shutil.copy(last_src, last_dst)
        print(f"  ✅ Saved last.pt for fold {fold} to Drive")
    # ──────────────────────────────────────────────────────────────────────────

    # Cleanup fold directories to save space
    shutil.rmtree(fold_dir)
    os.remove(fold_yaml_path)

# ─── FINAL SUMMARY ────────────────────────────────────────────────────────────
print(f"\n{'='*40}")
print("YOLOV8 K-FOLD FINAL RESULTS")
print(f"{'='*40}")
for r in fold_results:
    print(f"Fold {r['fold']}: mAP50={r['mAP50']:.4f} | "
          f"Precision={r['precision']:.4f} | "
          f"Recall={r['recall']:.4f}")

means = {
    "mAP50":     np.mean([r["mAP50"]     for r in fold_results]),
    "mAP50-95":  np.mean([r["mAP50-95"]  for r in fold_results]),
    "precision": np.mean([r["precision"] for r in fold_results]),
    "recall":    np.mean([r["recall"]    for r in fold_results]),
}
stds = {
    "mAP50":     np.std([r["mAP50"]     for r in fold_results]),
    "mAP50-95":  np.std([r["mAP50-95"]  for r in fold_results]),
    "precision": np.std([r["precision"] for r in fold_results]),
    "recall":    np.std([r["recall"]    for r in fold_results]),
}

print(f"\nMean ± Std across {K_FOLDS} folds:")
print(f"  mAP50:     {means['mAP50']:.4f} ± {stds['mAP50']:.4f}")
print(f"  mAP50-95:  {means['mAP50-95']:.4f} ± {stds['mAP50-95']:.4f}")
print(f"  Precision: {means['precision']:.4f} ± {stds['precision']:.4f}")
print(f"  Recall:    {means['recall']:.4f} ± {stds['recall']:.4f}")

# Save summary to Drive
summary_path = "/content/drive/MyDrive/RD2/yolo_kfold_results.txt"
with open(summary_path, "w") as f:
    for r in fold_results:
        f.write(f"Fold {r['fold']}: mAP50={r['mAP50']:.4f} | "
                f"Precision={r['precision']:.4f} | "
                f"Recall={r['recall']:.4f}\n")
    f.write(f"\nMean mAP50:     {means['mAP50']:.4f} ± {stds['mAP50']:.4f}\n")
    f.write(f"Mean mAP50-95:  {means['mAP50-95']:.4f} ± {stds['mAP50-95']:.4f}\n")
    f.write(f"Mean Precision: {means['precision']:.4f} ± {stds['precision']:.4f}\n")
    f.write(f"Mean Recall:    {means['recall']:.4f} ± {stds['recall']:.4f}\n")

print(f"\nResults saved to Google Drive at RD2/yolo_kfold_results.txt")

# ─── VERIFY ALL MODELS SAVED TO DRIVE ────────────────────────────────────────
print(f"\n{'='*40}")
print("MODELS SAVED TO DRIVE")
print(f"{'='*40}")
for fold in range(1, K_FOLDS + 1):
    path = f"{DRIVE_DIR}/yolo_fold{fold}_best.pt"
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} Fold {fold}: {path}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Total images found: 157

FOLD 1/5
  Train: 125 images
  Test:  32 images
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fold_1.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, f